# 03 · Stage 1 screening: transformer detectors


**Owner:** member C · **GPU:** ≈ 3.5 h (Deformable DETR) + 1.5 h (RT-DETR) + optional DETR · **Needs:** 00 · **Parallel with:** 01, 02

| Model | Implementation | Tier | Note |
|---|---|---|---|
| Deformable DETR | Hugging Face `SenseTime/deformable-detr` (COCO) | Essential | the paper's transformer; its ground-pretraining result (44.8 vs 33.2 AP) motivates notebook 07 |
| RT-DETR-L | Ultralytics `rtdetr-l.pt` (COCO) | Important | **replaces DINO** (see below) |
| DETR-R50 | Hugging Face `facebook/detr-resnet-50` | Optional | single-scale C5 features are known to fail on small objects; with 100 queries it cannot even output the ~78 objects/frame MAVREC has, so we use 300 queries (re-initialised query embeddings) — reported as a reference point only |

**Why no DINO?** The reference DINO-DETR code (IDEA-Research/DINO, detrex, MMDetection) needs custom CUDA ops (MultiScaleDeformableAttention / mmcv) compiled
for the exact torch/CUDA version; mmcv publishes no wheels for the torch versions Colab currently ships and the source build takes ~30–60 min
and often fails, and DINO-4scale at 960 px does not fit a T4 with a usable batch. RT-DETR is a DETR-family detector built for real-time use
(hybrid encoder + IoU-aware query selection, NMS-free) that trains on a T4 inside the same Ultralytics pipeline, so it is the most
defensible substitute. Report it as "RT-DETR (DINO substitute)", never as DINO.

Training details (HF models): AdamW, lr 1e-4 (backbone 1e-5), weight decay 1e-4, grad-clip 0.1, lr ×0.1 at 2/3 of training, **fp32**
(Hungarian matching is fragile in fp16; set `amp=True` in `make_spec` to trade stability for ~1.8× speed), images resized to 960×544.

In [3]:
# ===== 0. Setup — identical in every notebook. Runtime > Change runtime type > T4 GPU =====
SMOKE_TEST = False   # True: 16 train / 8 val images, 1 epoch — checks the whole pipeline in minutes
DRIVE_ROOT = "/content/drive/MyDrive/MAVREC"   # shared team folder (each member: "Add shortcut to Drive")

import os, sys, subprocess, importlib
try:
    from google.colab import drive
    try:
        os.listdir("/content/drive/MyDrive")   # raises if the Drive FUSE mount went stale
        drive.mount("/content/drive")
    except OSError:  # "Transport endpoint is not connected" after a Drive hiccup: remount
        drive.mount("/content/drive", force_remount=True)
except ImportError:
    print("Not in Colab — make sure DRIVE_ROOT exists locally.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.173", "transformers==5.18.0", "pycocotools>=2.0.8", "timm>=1.0.15", "tabulate", "tensorboard", "seaborn"], check=True)

CODE_DIR = os.path.join(DRIVE_ROOT, "Final_Project_Code")
assert os.path.exists(os.path.join(CODE_DIR, "mavrec_lib.py")), "Run 00_Setup_and_Data_Preparation first (it installs mavrec_lib.py)."
sys.path.insert(0, CODE_DIR)
import mavrec_lib as ml; importlib.reload(ml)
CFG = ml.setup(DRIVE_ROOT, smoke=SMOKE_TEST or None)
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
N_SMALL, N_SCREEN, N_LARGE = ml.TRAIN_SIZES      # 500, 1000, 2000

Mounted at /content/drive
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Environment:
  python       : 3.13.15
  torch        : 2.11.0+cu130
  cuda         : True
  torchvision  : 0.26.0+cu130
  ultralytics  : 8.4.173
  transformers : 5.18.0
  pycocotools  : 2.0.11
  numpy        : 2.1.3
  lib          : 1.0.0
  gpu          : Tesla T4
  gpu_mem_gb   : 14.6


### S1 · Deformable DETR @ 1000 / 960 — **Essential**
≈ 6–7 min/epoch × 30 epochs on a T4 (fp32). Interrupted runs resume from the last epoch.

In [ ]:
spec = ml.make_spec("deformable_detr", f"aerial_train_{N_SCREEN}", 960, stage="S1")
ml.run_experiment(spec)

[S1_deformable_detr_n1000_r960_full_COCO_s42] training deformable_detr | data=aerial_train_1000 | res=960 | epochs=30 | batch=2 | init=coco


Loading weights:   0%|          | 0/545 [00:00<?, ?it/s]

Resumed from epoch 3


/content/drive/MyDrive/MAVREC/Final_Project_Code/mavrec_lib.py:1817: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  losses.append(float(loss))


epoch 4/30 loss 1.7622 (458s)
epoch 5/30 loss 1.6813 (458s)
epoch 6/30 loss 1.6260 (457s)
epoch 7/30 loss 1.6237 (456s)
epoch 8/30 loss 1.5660 (457s)
epoch 9/30 loss 1.5504 (455s)
epoch 10/30 loss 1.5309 (456s)
epoch 11/30 loss 1.4862 (456s)
epoch 12/30 loss 1.4649 (456s)
epoch 13/30 loss 1.4520 (456s)
epoch 14/30 loss 1.4403 (456s)
epoch 15/30 loss 1.4338 (456s)
epoch 16/30 loss 1.4345 (457s)
epoch 17/30 loss 1.4163 (455s)
epoch 18/30 loss 1.3967 (455s)
epoch 19/30 loss 1.3899 (455s)
epoch 20/30 loss 1.4017 (455s)
epoch 21/30 loss 1.2839 (457s)
epoch 22/30 loss 1.2076 (454s)
epoch 23/30 loss 1.1874 (455s)
epoch 24/30 loss 1.1674 (455s)
epoch 25/30 loss 1.1547 (453s)
epoch 26/30 loss 1.1357 (456s)
epoch 27/30 loss 1.1136 (456s)
epoch 28/30 loss 1.0997 (455s)
epoch 29/30 loss 1.0908 (454s)
epoch 30/30 loss 1.0887 (456s)


Loading weights:   0%|          | 0/545 [00:00<?, ?it/s]

  predicted 538 images (full) in 84s, 161400 detections
[S1_deformable_detr_n1000_r960_full_COCO_s42] val|full|r960: mAP50-95=19.566 AP50=38.905 APS=7.411 APt=2.254 R=53.982 FPS=7.01


{'experiment_id': 'S1_deformable_detr_n1000_r960_full_COCO_s42',
 'stage': 'S1',
 'model_family': 'Transformer',
 'model_version': 'deformable_detr',
 'backbone': 'ResNet-50 (multi-scale)',
 'dataset_size': 1000,
 'train_set': 'aerial_train_1000',
 'resolution': 960,
 'tiling': 'none',
 'pretraining': 'COCO',
 'init': 'coco',
 'epochs': 30,
 'seed': 42,
 'notes': '',
 'batch_size': 2,
 'train_time_h': 3.84,
 'GPU_memory_train_GB': 6.25,
 'checkpoint_path': '/content/drive/MyDrive/MAVREC/Final_Project_Results/runs/S1_deformable_detr_n1000_r960_full_COCO_s42/final.pt',
 'spec': {'stage': 'S1',
  'model_version': 'deformable_detr',
  'model_family': 'Transformer',
  'framework': 'hf',
  'backbone': 'ResNet-50 (multi-scale)',
  'train_set': 'aerial_train_1000',
  'val_set': 'aerial_val',
  'test_set': 'aerial_test',
  'train_pack': 'aerial_1280',
  'eval_pack': 'aerial_1280',
  'native_pack': 'aerial_native',
  'dataset_size': 1000,
  'resolution': 960,
  'train_res': 960,
  'epochs': 30,


### S1 · RT-DETR-L @ 1000 / 960 (DINO substitute) — **Important**

In [4]:
spec = ml.make_spec("rtdetr-l", f"aerial_train_{N_SCREEN}", 960, stage="S1")
ml.run_experiment(spec)

Pack 'aerial_1280' unpacked to local disk in 73s
[S1_rtdetr-l_n1000_r960_full_COCO_s42] training rtdetr-l | data=aerial_train_1000 | res=960 | epochs=40 | batch=4 | init=coco
Resuming from /content/drive/MyDrive/MAVREC/Final_Project_Results/runs/S1_rtdetr-l_n1000_r960_full_COCO_s42/train/weights/last.pt
New https://pypi.org/project/ultralytics/8.4.174 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/mavrec_local/yolo/aerial_train_1000__aerial_val__aerial_1280/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      28/40       6.5G     0.9492     0.3322     0.1199        714        960: 1% ──────────── 3/250 1.2it/s 7.2s<3:30

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      28/40      6.51G     0.7635     0.3605    0.07785        284        960: 4% ╸─────────── 11/250 1.4it/s 12.4s<2:47

...

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      40/40      6.51G     0.5662      0.369    0.02767        196        960: 99% ━━━━━━━━━━━╸ 248/250 1.8it/s 2:32<1.1s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      40/40      6.51G     0.5661     0.3691    0.02767        344        960: 100% ━━━━━━━━━━━━ 250/250 1.6it/s 2:33
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 68/68 2.4it/s 28.9s
                   all        538      42927      0.764      0.847      0.824      0.536

13 epochs completed in 0.587 hours.
Optimizer stripped from /content/drive/.shortcut-targets-by-id/1SXwOfZA3KCmwUod3NvTUhhBmdmxyqjpH/MAVREC/Final_Project_Results/runs/S1_rtdetr-l_n1000_r960_full_COCO_s42/train/weights/last.pt, 66.3MB
Optimizer stripped from /content/drive/.shortcut-targets-by-id/1SXwOfZA3KCmwUod3NvTUhhBmdmxyqjpH/MAVREC/Final_Project_Results/runs/S1_rtdetr-l_n1000_r960_full_COCO_s42/train/weights/best.pt, 66.3MB

Validating /content/drive/.shortcut-targets-by-id/1SXwOfZA3KCmwUod3NvTUhhBmdmxyqjpH/MAVREC/Final_Project_Results/runs/S1_rtdetr-l_n1000_r960_full_COCO_s42/train/weights/best.pt...
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+c

{'experiment_id': 'S1_rtdetr-l_n1000_r960_full_COCO_s42',
 'stage': 'S1',
 'model_family': 'Transformer',
 'model_version': 'rtdetr-l',
 'backbone': 'HGNetv2-L',
 'dataset_size': 1000,
 'train_set': 'aerial_train_1000',
 'resolution': 960,
 'tiling': 'none',
 'pretraining': 'COCO',
 'init': 'coco',
 'epochs': 40,
 'seed': 42,
 'notes': '',
 'batch_size': 4,
 'train_time_h': 0.602,
 'GPU_memory_train_GB': 7.16,
 'checkpoint_path': '/content/drive/MyDrive/MAVREC/Final_Project_Results/runs/S1_rtdetr-l_n1000_r960_full_COCO_s42/train/weights/last.pt',
 'spec': {'stage': 'S1',
  'model_version': 'rtdetr-l',
  'model_family': 'Transformer',
  'framework': 'ultralytics',
  'backbone': 'HGNetv2-L',
  'train_set': 'aerial_train_1000',
  'val_set': 'aerial_val',
  'test_set': 'aerial_test',
  'train_pack': 'aerial_1280',
  'eval_pack': 'aerial_1280',
  'native_pack': 'aerial_native',
  'dataset_size': 1000,
  'resolution': 960,
  'train_res': 960,
  'epochs': 40,
  'batch_size': 4,
  'seed': 42,


### S1 · DETR-R50 (reference only) — **Optional**

In [6]:
RUN_DETR = False
if RUN_DETR:
    spec = ml.make_spec("detr_r50", f"aerial_train_{N_SCREEN}", 960, stage="S1", notes="300 queries; expected weak on small objects")
    ml.run_experiment(spec)

## Results (transformer family)

In [5]:
df = ml.results_df(stage="S1")
t = df[df.model_family == "Transformer"]
display(t[["experiment_id","stage","model_version","dataset_size","resolution","tiling","inference_mode","pretraining","mAP50_95","mAP50","APS","APt","APM","recall","F1","FPS","GPU_memory_infer_GB","parameters_M","train_time_h"]].sort_values("mAP50_95", ascending=False))
ml.save_table(t[["experiment_id","stage","model_version","dataset_size","resolution","tiling","inference_mode","pretraining","mAP50_95","mAP50","APS","APt","APM","recall","F1","FPS","GPU_memory_infer_GB","parameters_M","train_time_h"]], "S1_transformers")

,experiment_id,stage,model_version,dataset_size,resolution,tiling,inference_mode,pretraining,mAP50_95,mAP50,APS,APt,APM,recall,F1,FPS,GPU_memory_infer_GB,parameters_M,train_time_h
1,S1_rtdetr-l_n1000_r960_full_COCO_s42,S1,rtdetr-l,1000,960.0,none,full,COCO,52.887,82.186,34.665,13.489,49.767,86.293,88.490,13.51,0.25,32.83,0.602
0,S1_deformable_detr_n1000_r960_full_COCO_s42,S1,deformable_detr,1000,960.0,none,full,COCO,19.566,38.905,7.411,2.254,26.717,53.982,59.833,7.01,0.87,40.05,3.840


saved /content/drive/MyDrive/MAVREC/Final_Project_Results/tables/S1_transformers.csv


'/content/drive/MyDrive/MAVREC/Final_Project_Results/tables/S1_transformers'